🔄 ch10 > sec02 > 01_conditional_tools.ipynb 예제 재사용 하기

#### hello 도구 테스트

In [2]:
# https://github.com/langchain-ai/langchain-mcp-adapters?tab=readme-ov-file#streamable-http

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

from langchain_mcp_adapters.tools import load_mcp_tools # uv add langchain-mcp-adapters

async with streamablehttp_client("http://127.0.0.1:8000/mcp/") as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()

        # Get tools
        tools = await load_mcp_tools(session)

        # 'hello' 도구를 비동기적으로 실행하고 결과를 result 변수에 할당
        # MCP는 서버이기 때문에 네트워크로 비동기 통신함
        result = await tools[0].ainvoke({"name": "김일남"})
        print(result)


[{'type': 'text', 'text': '안녕하세요, 김일남님!', 'id': 'lc_e79fe03f-8c55-4204-8173-531d087f5129'}]


#### 랭그래프에서 MCP 연동

> Use MCP: <https://langchain-ai.github.io/langgraph/agents/mcp/#use-mcp-tools> </br>
> GitHub: <https://github.com/langchain-ai/langchain-mcp-adapters?tab=readme-ov-file#using-with-langgraph-stategraph>

In [36]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "mcp_tools": {
            "url": "http://localhost:8000/mcp",
            "transport": "streamable_http",
        }
    }
)
tools = await client.get_tools()

In [37]:
print(f"로드된 도구들: {[t.name for t in tools]}") 

로드된 도구들: ['hello', 'get_current_time', 'get_yf_stock_history']


In [38]:
tools

[StructuredTool(name='hello', description='간단한 인사말을 반환하는 도구', args_schema={'properties': {'name': {'default': '아무개', 'title': 'Name', 'type': 'string'}}, 'title': 'helloArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000023E322C6480>),
 StructuredTool(name='get_current_time', description=" 현재 시각을 반환하는 함수\n\n    Args:\n        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함\n        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨\n    ", args_schema={'properties': {'timezone': {'default': 'Asia/Seoul', 'title': 'Timezone', 'type': 'string'}, 'location': {'default': '부산', 'title': 'Location', 'type': 'string'}}, 'title': 'get_current_timeArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000023E322C7EC0>),
 StructuredTool(name='get_yf_stock_history', description=' 주식

In [39]:
# 에이전트 생성 전 출력해보기
print(f"로드된 도구 개수: {len(tools)}")
for t in tools:
    print(f"- {t.name}: {t.description}")

로드된 도구 개수: 3
- hello: 간단한 인사말을 반환하는 도구
- get_current_time:  현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    
- get_yf_stock_history:  주식 종목의 가격 데이터를 조회하는 함수


In [40]:
result = await tools[0].ainvoke({"name": "김일남"})
result

[{'type': 'text',
  'text': '안녕하세요, 김일남님!',
  'id': 'lc_aba24f5f-099c-43a4-8a22-9d27e0225d7b'}]

In [41]:
result = await tools[1].ainvoke({"timezone": "Asia/Seoul", "location": "부산"})
result

[{'type': 'text',
  'text': 'Asia/Seoul (부산) 현재시각 2026-01-25 02:41:39 ',
  'id': 'lc_303daf25-3c2e-4f1c-8a06-a9a770910d99'}]

In [46]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [47]:
from dataclasses import dataclass

@dataclass
class UserContext:
    user_id: str

In [51]:
system_prompt = (
    "너는 사용자의 질문에 답변을 하기 위해 도구를 사용할 수 있는 비서다. "
    "지역에 대한 시간을 물어보면, 해당 지역의 타임존(예: 서울/부산은 'Asia/Seoul')을 "
    "스스로 추론하여 get_current_time 도구를 즉시 호출해라."
)

In [52]:
from langchain.agents import create_agent

# 에이전트 생성
agent = create_agent(
    model,
    tools=tools,
    context_schema=UserContext,
    system_prompt=system_prompt
)

In [53]:
# 비동기 호출 요망
result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "부산은 지금 몇시야?"}]},
    context=UserContext(user_id="user123")
)

In [54]:
result

{'messages': [HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}, id='cb810237-8f89-4789-b235-61f5f0ec6f36'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "Asia/Seoul", "location": "\\ubd80\\uc0b0"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--42f0bc4e-9996-440e-94b4-19f54e46d8bc-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': '3c18bae4-1231-438b-ba69-e4d9c190cf7c', 'type': 'tool_call'}], usage_metadata={'input_tokens': 381, 'output_tokens': 24, 'total_tokens': 405, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content=[{'type': 'text', 'text': 'Asia/Seoul (부산) 현재시각 2026-01-25 02:45:16 ', 'id': 'lc_7eae78f7-2ad9-478b-a61d-5c2eed32b063'}], name='get_cu